### Підготовка до роботи та Завдання 1. Зчитування і Data Cleaning
Звантажити та відкрити датасет: Individual Household Electric Power Consumption Dataset. Здійснити data cleaning (обробити пропущені значення). Датасет має знаходитись локально у файлі `household_power_consumption.txt`.

In [1]:
import pandas as pd
import numpy as np
import timeit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from IPython.display import display

print("Завантажуємо датасет та виконуємо Data Cleaning")
print("Перетворюємо символи '?' на порожні значення та видаляємо рядки з пропусками.")

df_power = pd.read_csv('household_power_consumption.txt', sep=';', na_values=['?'], low_memory=False)
df_power.dropna(inplace=True)

print("Дані очищено. Загальна кількість записів")
print(len(df_power))
display(df_power.head(3))

Завантажуємо датасет та виконуємо Data Cleaning
Перетворюємо символи '?' на порожні значення та видаляємо рядки з пропусками.
Дані очищено. Загальна кількість записів
2049280


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0


### Завдання 2.1. Вибірка за потужністю
Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт. Профілювання часу виконання здійснюється за допомогою модуля timeit.

In [2]:
def filter_high_active_power(dataframe):
    filtered_data = dataframe[dataframe['Global_active_power'] > 5.0]
    return filtered_data

print("Результат вибірки Потужність > 5 кВт")
result_high_power = filter_high_active_power(df_power)
display(result_high_power.head(3))

print("Час виконання функції (секунди, 10 повторень)")
execution_time_task1 = timeit.timeit(lambda: filter_high_active_power(df_power), number=10)
print(execution_time_task1)

Результат вибірки Потужність > 5 кВт


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0


Час виконання функції (секунди, 10 повторень)
0.0402020001783967


### Завдання 2.2. Вибірка за силою струму та споживанням груп
Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильник (Sub_metering_2) споживають більше, ніж бойлер та кондиціонер (Sub_metering_3).

In [3]:
def filter_current_and_appliances(dataframe):
    condition_current_min = dataframe['Global_intensity'] >= 19.0
    condition_current_max = dataframe['Global_intensity'] <= 20.0
    condition_appliances = dataframe['Sub_metering_2'] > dataframe['Sub_metering_3']
    
    filtered_data = dataframe[condition_current_min & condition_current_max & condition_appliances]
    return filtered_data

print("Результат вибірки Струм 19-20 А, Група 2 > Групи 3")
result_appliances = filter_current_and_appliances(df_power)
display(result_appliances.head(3))

print("Час виконання функції (секунди, 10 повторень)")
execution_time_task2 = timeit.timeit(lambda: filter_current_and_appliances(df_power), number=10)
print(execution_time_task2)

Результат вибірки Струм 19-20 А, Група 2 > Групи 3


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0


Час виконання функції (секунди, 10 повторень)
0.088310900144279


### Завдання 2.3. Випадкова вибірка та середні величини
Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії.

In [4]:
def sample_and_calculate_means(dataframe):
    random_sample = dataframe.sample(n=500000, replace=False, random_state=42)
    means = random_sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    return means

print("Середні величини 3-х груп для випадкових 500 000 записів")
result_means = sample_and_calculate_means(df_power)
print(result_means)

print("Час виконання функції (секунди, 10 повторень)")
execution_time_task3 = timeit.timeit(lambda: sample_and_calculate_means(df_power), number=10)
print(execution_time_task3)

Середні величини 3-х груп для випадкових 500 000 записів
Sub_metering_1    1.119258
Sub_metering_2    1.308912
Sub_metering_3    6.452950
dtype: float64
Час виконання функції (секунди, 10 повторень)
1.4509189003147185


### Завдання 2.4. Складна вечірня вибірка
Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання припадає на 2 групу. Потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.

In [5]:
def complex_evening_filter(dataframe):
    condition_time = dataframe['Time'] >= '18:00:00'
    condition_power = dataframe['Global_active_power'] > 6.0
    
    condition_group2_beats_1 = dataframe['Sub_metering_2'] > dataframe['Sub_metering_1']
    condition_group2_beats_3 = dataframe['Sub_metering_2'] > dataframe['Sub_metering_3']
    
    filtered_data = dataframe[condition_time & condition_power & condition_group2_beats_1 & condition_group2_beats_3]
    
    half_index = len(filtered_data) // 2
    
    first_half = filtered_data.iloc[:half_index]
    second_half = filtered_data.iloc[half_index:]
    
    first_half_sampled = first_half.iloc[::3]
    second_half_sampled = second_half.iloc[::4]
    
    final_result = pd.concat([first_half_sampled, second_half_sampled])
    return final_result

print("Результат складної вечірньої вибірки")
result_complex = complex_evening_filter(df_power)
display(result_complex.head(3))

print("Час виконання функції (секунди, 10 повторень)")
execution_time_task4 = timeit.timeit(lambda: complex_evening_filter(df_power), number=10)
print(execution_time_task4)

Результат складної вечірньої вибірки


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0
17494,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0


Час виконання функції (секунди, 10 повторень)
1.0933177000842988


### Завдання 3. Нормування та стандартизація
Пронормувати та стандартизувати вибраний датасет (на прикладі числових колонок). Використовуються MinMaxScaler та StandardScaler з бібліотеки scikit-learn.

In [6]:
print("Беремо копію датафрейму для нормалізації та стандартизації")
numeric_columns = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity']

df_to_scale = df_power[numeric_columns].copy()

scaler_minmax = MinMaxScaler()
normalized_data = scaler_minmax.fit_transform(df_to_scale)
df_normalized = pd.DataFrame(normalized_data, columns=numeric_columns)

scaler_standard = StandardScaler()
standardized_data = scaler_standard.fit_transform(df_to_scale)
df_standardized = pd.DataFrame(standardized_data, columns=numeric_columns)

print("Пронормовані дані MinMaxScaler, значення від 0 до 1")
display(df_normalized.head(3))

print("Стандартизовані дані StandardScaler, середнє = 0, дисперсія = 1")
display(df_standardized.head(3))

Беремо копію датафрейму для нормалізації та стандартизації
Пронормовані дані MinMaxScaler, значення від 0 до 1


,Global_active_power,Global_reactive_power,Voltage,Global_intensity
0,0.374796,0.300719,0.376090,0.377593
1,0.478363,0.313669,0.336995,0.473029
2,0.479631,0.358273,0.326010,0.473029


Стандартизовані дані StandardScaler, середнє = 0, дисперсія = 1


,Global_active_power,Global_reactive_power,Voltage,Global_intensity
0,2.955077,2.610721,-1.851816,3.098789
1,4.037085,2.770406,-2.225274,4.133800
2,4.050326,3.320432,-2.330213,4.133800


### Завдання 4. Коефіцієнти кореляції
Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів. Обрано колонки загальної активної потужності (Global_active_power) та сили струму (Global_intensity).

In [7]:
pearson_corr = df_power['Global_active_power'].corr(df_power['Global_intensity'], method='pearson')
spearman_corr = df_power['Global_active_power'].corr(df_power['Global_intensity'], method='spearman')

print("Коефіцієнт кореляції Пірсона")
print(pearson_corr)

print("Коефіцієнт кореляції Спірмена")
print(spearman_corr)

Коефіцієнт кореляції Пірсона
0.9988886002095853
Коефіцієнт кореляції Спірмена
0.9953719367299743


### Завдання 5. One Hot Encoding
Провести One Hot Encoding категоріального атрибута. Створюємо нову категоріальну колонку 'Hour' (година) з колонки 'Time' та застосовуємо метод get_dummies.


In [8]:
def apply_refined_ohe(dataframe):
    df_refined = dataframe.copy()
    
    df_refined['Period'] = df_refined['Time'].apply(lambda x: 'AM' if x < '12:00:00' else 'PM')
    
    ohe_period = pd.get_dummies(df_refined['Period'], prefix='Period')
    
    return pd.concat([df_refined[['Time', 'Period']], ohe_period], axis=1)

print("One Hot Encoding для періодів AM та PM")
df_ohe_result = apply_refined_ohe(df_power)
display(df_ohe_result.head(10))

One Hot Encoding для періодів AM та PM


,Time,Period,Period_AM,Period_PM
0,17:24:00,PM,False,True
1,17:25:00,PM,False,True
2,17:26:00,PM,False,True
3,17:27:00,PM,False,True
4,17:28:00,PM,False,True
5,17:29:00,PM,False,True
6,17:30:00,PM,False,True
7,17:31:00,PM,False,True
8,17:32:00,PM,False,True
9,17:33:00,PM,False,True
